# 08 — Master tables and evidence-based report

This notebook performs no training. It verifies upstream manifests,
consolidates actual CSV results, copies final figures, writes a Markdown
report, and creates a lightweight evidence ZIP. It never invents or
manually types a numerical result.

In [1]:
import json, os, shutil, zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

PROFILE = os.getenv("REVALIDATION_PROFILE", "research").strip().lower()

In [2]:
from pathlib import Path

def resolve_project_root() -> Path:
    starts = [Path.cwd().resolve()]
    try:
        starts.append(Path(__file__).resolve().parent)
    except NameError:
        pass
    for start in starts:
        for candidate in [start, *start.parents]:
            if candidate.name == "Marco_Revalidation_v2":
                return candidate
            nested = candidate / "Marco_Revalidation_v2"
            if (nested / "00_protocol").exists():
                return nested
    raise FileNotFoundError(
        "Could not locate Marco_Revalidation_v2. Run this notebook from inside "
        "the extracted project directory."
    )

PROJECT_ROOT = resolve_project_root()
CONTRACT_DIR = PROJECT_ROOT / "01_data_contract"
CLASSICAL_DIR = PROJECT_ROOT / "02_classical_augmentation" / "outputs"
CTGAN_DIR = PROJECT_ROOT / "03_ctgan"
QGAN_DIR = PROJECT_ROOT / "04_qgan"
FIDELITY_DIR = PROJECT_ROOT / "05_fidelity"
DOWNSTREAM_DIR = PROJECT_ROOT / "06_downstream"
HANDOFF_DIR = PROJECT_ROOT / "07_quantum_handoff"
REPORT_DIR = PROJECT_ROOT / "08_reporting"
print("Project root:", PROJECT_ROOT)

Project root: C:\Users\HP\Desktop\Qintern\week5\Marco_Revalidation_v2


## 1. Hard gate: every upstream stage must be complete

In [3]:
OUTPUT_DIR = REPORT_DIR
FIGURE_DIR = REPORT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

required = {
    "data": CONTRACT_DIR / "dataset_manifest.json",
    "classical": CLASSICAL_DIR / "augmentation_manifest.json",
    "ctgan": CTGAN_DIR / "outputs" / "ctgan_manifest.json",
    "qgan": QGAN_DIR / "outputs" / "qgan_manifest.json",
    "fidelity": FIDELITY_DIR / "outputs" / "fidelity_manifest.json",
    "downstream": DOWNSTREAM_DIR / "outputs" / "downstream_decision.json",
    "handoff": HANDOFF_DIR / "outputs" / "quantum_handoff_manifest.json",
}
missing = [str(p) for p in required.values() if not p.exists()]
if missing:
    raise FileNotFoundError("Incomplete pipeline. Missing:\n- " + "\n- ".join(missing))
manifests = {name: json.loads(path.read_text(encoding="utf-8"))
             for name, path in required.items()}
hashes = {
    m.get("dataset_sha256") or m.get("source_sha256") or m.get("input_dataset_sha256")
    for m in manifests.values()
}
if hashes != {"cc7a637a174ffe797e0af0375bce3c09561f0dc8b8115c0a6292718034f5012a"}:
    raise RuntimeError(f"Upstream dataset hash disagreement: {hashes}")
profiles = {m.get("profile") for m in manifests.values() if m.get("profile")}
reportable = profiles == {"research"} and PROFILE == "research"
print("Profiles found:", profiles, "Final reportable:", reportable)

Profiles found: {'research'} Final reportable: True


## 2. Consolidate fidelity and downstream utility as separate axes

In [4]:
fidelity = pd.read_csv(FIDELITY_DIR / "outputs" / "common_fidelity_master_table.csv")
downstream = pd.read_csv(DOWNSTREAM_DIR / "outputs" / "classifier_results_mean_std.csv")
evaluation = downstream[downstream.partition.eq("fixed_evaluation")].copy()
best_downstream = evaluation.sort_values("macro_f1_mean", ascending=False).groupby(
    "method", as_index=False
).first()

fidelity_method_map = {"Original real train": "Original"}
fidelity["method"] = fidelity.method.replace(fidelity_method_map)
columns = ["method", "mean_ks_mean", "mean_wasserstein_mean",
           "rbf_mmd_squared_mean", "mean_rank"]
master = fidelity[columns].merge(
    best_downstream[["method", "classifier", "macro_f1_mean", "macro_f1_std",
                     "mcc_mean", "f1_trojan_mean"]],
    on="method", how="outer", validate="one_to_one",
)
original_macro_by_classifier = evaluation[evaluation.method.eq("Original")].set_index(
    "classifier"
)["macro_f1_mean"]
master["macro_f1_delta_vs_original_same_classifier"] = [
    row.macro_f1_mean - original_macro_by_classifier.get(row.classifier, np.nan)
    for row in master.itertuples()
]
master.to_csv(REPORT_DIR / "master_augmentation_comparison.csv", index=False)
display(master.sort_values("macro_f1_mean", ascending=False))

,method,mean_ks_mean,mean_wasserstein_mean,rbf_mmd_squared_mean,mean_rank,classifier,macro_f1_mean,macro_f1_std,mcc_mean,f1_trojan_mean,macro_f1_delta_vs_original_same_classifier
3,Original,0.040687,0.065713,0.000403,1.0,LightGBM,0.805054,0.001489,0.806228,0.722773,0.000000
0,ADASYN,0.067992,0.085556,0.002062,3.0,Random Forest,0.804399,0.000489,0.805522,0.717063,0.006725
4,QGAN stabilized,0.355305,0.405348,0.070987,6.0,LightGBM,0.803459,0.000510,0.804676,0.720678,-0.001595
1,Borderline-SMOTE,0.076490,0.103188,0.004183,4.0,Random Forest,0.802729,0.000041,0.803666,0.717797,0.005056
5,SMOTE,0.048273,0.067101,0.000501,2.0,Random Forest,0.798225,0.001089,0.799126,0.712941,0.000552
2,CTGAN,0.308718,0.160132,0.009041,5.0,Random Forest,0.798137,0.000446,0.799014,0.710846,0.000464


## 3. Configuration and tables

In [5]:
config_rows = [
    {"stage": "Data contract", "selection_data": "none", "test_scored": False,
     "artifact": "locked split + preprocessing"},
    {"stage": "Classical augmentation", "selection_data": "validation", "test_scored": False,
     "artifact": "SMOTE / Borderline-SMOTE / ADASYN"},
    {"stage": "CTGAN", "selection_data": "validation", "test_scored": False,
     "artifact": f"epoch {manifests['ctgan']['selected_epochs']}, budget {manifests['ctgan']['selected_budget']:.0%}"},
    {"stage": "QGAN", "selection_data": "validation", "test_scored": False,
     "artifact": f"6 qubits, 2 layers, budget {manifests['qgan']['selected_budget']:.0%}"},
    {"stage": "Fidelity", "selection_data": "validation reference", "test_scored": False,
     "artifact": "KS / Wasserstein / RBF-MMD"},
    {"stage": "Downstream", "selection_data": "validation", "test_scored": True,
     "artifact": "RF / XGBoost / LightGBM / Linear SVM"},
    {"stage": "Quantum handoff", "selection_data": "shared validation", "test_scored": False,
     "artifact": "n=200/1000; canonical q=6, d=12"},
]
config = pd.DataFrame(config_rows)
config.to_csv(REPORT_DIR / "master_configuration_table.csv", index=False)

figure_sources = {
    "fidelity_comparison.png": FIDELITY_DIR / "outputs" / "01_common_fidelity_comparison.png",
    "fidelity_pca_overlap.png": FIDELITY_DIR / "outputs" / "03_pca_overlap.png",
    "downstream_macro_trojan.png": DOWNSTREAM_DIR / "outputs" / "01_macro_and_trojan_f1.png",
    "downstream_deltas.png": DOWNSTREAM_DIR / "outputs" / "02_improvement_over_original.png",
    "selected_confusion_matrix.png": DOWNSTREAM_DIR / "outputs" / "03_selected_confusion_matrix.png",
    "qgan_diagnostics.png": QGAN_DIR / "outputs" / "01_qgan_training_diagnostics.png",
}
for name, source in figure_sources.items():
    if source.exists():
        shutil.copy2(source, FIGURE_DIR / name)

## 4. Generate the factual Markdown report

In [ ]:
best_utility = master.sort_values("macro_f1_mean", ascending=False).iloc[0]
best_fidelity = master.sort_values("mean_rank", ascending=True).iloc[0]
decision = manifests["downstream"]
selected_eval = evaluation[
    evaluation.method.eq(decision["selected_method"])
    & evaluation.classifier.eq(decision["selected_classifier"])
].iloc[0]

method_lines = []
for row in master.sort_values("macro_f1_mean", ascending=False).itertuples():
    method_lines.append(
        f"- **{row.method}** — best fixed-evaluation classifier: {row.classifier}; "
        f"Macro-F1 {row.macro_f1_mean:.4f} ± {row.macro_f1_std:.4f}; "
        f"MCC {row.mcc_mean:.4f}; Trojan F1 {row.f1_trojan_mean:.4f}; "
        f"mean fidelity rank {row.mean_rank:.2f}."
    )
profile_warning = "" if reportable else (
    "\n> **NON-REPORTABLE RUN:** at least one stage used the smoke profile. "
    "Re-run all stages with `--profile research`.\n"
)
report = "\n".join([
    "# Marco Revalidation v2 — Final Technical Report", "",
    f"Generated: {datetime.now(timezone.utc).isoformat()}", profile_warning,
    "## Scope and locked protocol", "",
    "The analysis revalidates augmentation on CIC-MalMem-2022 from the canonical "
    f"SHA-256 `{manifests['data']['source_sha256']}`. After removing 534 exact "
    "full-record duplicates, the locked dataset contains 58,062 rows: 29,231 "
    "Benign, 9,529 Ransomware, 9,815 Spyware, and 9,487 Trojan. The fixed split "
    "is 40,642 / 8,710 / 8,710 (train / validation / fixed evaluation), seed 42. "
    "All later stages load the same row assignments and 52-feature train-only preprocessing.",
    "",
    "Twelve feature-equivalent groups cross partitions. The split is retained for "
    "comparability and the limitation is disclosed. The final partition was inspected "
    "in older iterations, so it is described as a fixed evaluation partition reused "
    "for protocol reconciliation.", "",
    "## Augmentation methods", "",
    "Classical interpolation includes SMOTE, Borderline-SMOTE, and ADASYN. CTGAN was "
    "retrained on every locked training row with independent epoch-budget restarts; "
    f"the selected checkpoint used {manifests['ctgan']['selected_epochs']} epochs. "
    "The stabilized QGAN used 6 qubits, 8 PCA components, two data-reuploading layers, "
    "ring-CNOT entanglement, analytic simulation, moment regularization, small "
    "initialization, and training-only output-bias calibration.", "",
    "## Fidelity", "",
    "One common bootstrap implementation evaluated featurewise KS, featurewise "
    "Wasserstein, and RBF-MMD against real validation rows. The best mean fidelity "
    f"rank was **{best_fidelity.method}** ({best_fidelity.mean_rank:.2f}). Fidelity "
    "is reported separately from classifier utility; no causal equivalence is assumed.", "",
    "## Downstream classifier results", "",
    f"Validation selected **{decision['selected_method']} + {decision['selected_classifier']}**. "
    f"Its fixed-evaluation Macro-F1 was {selected_eval.macro_f1_mean:.4f} ± "
    f"{selected_eval.macro_f1_std:.4f}, MCC was {selected_eval.mcc_mean:.4f}, and "
    f"Trojan F1 was {selected_eval.f1_trojan_mean:.4f}. Across each method's best "
    "classifier, the highest fixed-evaluation Macro-F1 was "
    f"**{best_utility.method} + {best_utility.classifier}** at "
    f"{best_utility.macro_f1_mean:.4f} ± {best_utility.macro_f1_std:.4f}.", "",
    *method_lines, "",
    "## Quantum handoff", "",
    "The package exports balanced n=200 and n=1000 training sets in a PCA basis fit "
    "only on original real training data. The canonical convention is q=6 and 12 "
    "features (two per qubit); optional q=4/8/12 files support capacity analysis. "
    "All methods share identical validation rows. No fixed-evaluation rows or quantum "
    "outcome claims are included in the handoff.", "",
    "## Conclusions and limitations", "",
    "1. Fidelity and downstream utility are distinct axes and are interpreted separately.",
    "2. The QGAN result is simulator-based and uses a declared per-class training cap.",
    "3. CTGAN checkpoint candidates are independent deterministic restarts.",
    "4. A group-aware sensitivity split is required before a publication-level claim.",
    "5. No model was retuned after fixed-evaluation results were computed.",
])
(REPORT_DIR / "FINAL_REPORT.md").write_text(report, encoding="utf-8")
print(report)

# Marco Revalidation v2 — Final Technical Report

Generated: 2026-08-11T23:23:29.519730+00:00

## Scope and locked protocol

The analysis revalidates augmentation on CIC-MalMem-2022 from the canonical SHA-256 `cc7a637a174ffe797e0af0375bce3c09561f0dc8b8115c0a6292718034f5012a`. After removing 534 exact full-record duplicates, the locked dataset contains 58,062 rows: 29,231 Benign, 9,529 Ransomware, 9,815 Spyware, and 9,487 Trojan. The fixed split is 40,642 / 8,710 / 8,710 (train / validation / fixed evaluation), seed 42. All later stages load the same row assignments and 52-feature train-only preprocessing.

Twelve feature-equivalent groups cross partitions. The split is retained for comparability and the limitation is disclosed. The final partition was inspected in older iterations, so it is described as a fixed evaluation partition reused for protocol reconciliation.

## Augmentation methods

Classical interpolation includes SMOTE, Borderline-SMOTE, and ADASYN. CTGAN was retrained on e

## 5. Package lightweight evidence

In [7]:
evidence_zip = REPORT_DIR / "Marco_Revalidation_v2_evidence.zip"
allowed = {".json", ".csv", ".md", ".png", ".txt", ".ipynb"}
excluded_parts = {"data", "checkpoints", ".venv", ".ipynb_checkpoints"}
with zipfile.ZipFile(evidence_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in PROJECT_ROOT.rglob("*"):
        if not path.is_file() or path == evidence_zip:
            continue
        if any(part in excluded_parts for part in path.relative_to(PROJECT_ROOT).parts):
            continue
        if path.suffix.lower() not in allowed:
            continue
        archive.write(path, path.relative_to(PROJECT_ROOT))
final_manifest = {
    "protocol_version": "revalidation_v2_final_report_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "reportable": reportable, "profiles_found": sorted(profiles),
    "dataset_sha256": manifests["data"]["source_sha256"],
    "master_rows": len(master), "evidence_zip": evidence_zip.name,
    "selected_downstream": {
        "method": decision["selected_method"],
        "classifier": decision["selected_classifier"],
        "fixed_evaluation_macro_f1_mean": float(selected_eval.macro_f1_mean),
    },
}
(REPORT_DIR / "final_report_manifest.json").write_text(
    json.dumps(final_manifest, indent=2), encoding="utf-8"
)
print("REPORTING STAGE ACCEPTED", final_manifest)

REPORTING STAGE ACCEPTED {'protocol_version': 'revalidation_v2_final_report_v1', 'created_at_utc': '2026-08-11T23:23:30.402637+00:00', 'reportable': True, 'profiles_found': ['research'], 'dataset_sha256': 'cc7a637a174ffe797e0af0375bce3c09561f0dc8b8115c0a6292718034f5012a', 'master_rows': 6, 'evidence_zip': 'Marco_Revalidation_v2_evidence.zip', 'selected_downstream': {'method': 'ADASYN', 'classifier': 'Random Forest', 'fixed_evaluation_macro_f1_mean': 0.8043987365435673}}
